In [0]:
pop_df = spark.table("workspace.default.population_growth_rate")
inflation_df = spark.table("workspace.default.inflation_rate")
interest_df = spark.table("workspace.default.interest_rate")

In [0]:
from pyspark.sql.functions import col

pop_df = pop_df.withColumnRenamed("Year", "year") \
               .withColumnRenamed("% Increase in Population", "population_growth")

pop_df = pop_df.select("year", "population_growth")

display(pop_df)

year,population_growth
1950,0.00%
1951,2.21%
1952,2.21%
1953,2.21%
1954,2.23%
1955,2.27%
1956,2.28%
1957,2.28%
1958,2.25%
1959,2.25%


In [0]:
from pyspark.sql.functions import to_date, year, avg

inflation_df = inflation_df.withColumn(
    "date",
    to_date(col("observation_date"), "dd-MM-yyyy")
)

inflation_df = inflation_df.withColumn(
    "year",
    year(col("date"))
)

inflation_df = inflation_df.withColumnRenamed("Inflation rate", "inflation_rate")

In [0]:
inflation_df = inflation_df.groupBy("year") \
    .agg(avg("inflation_rate").alias("inflation_rate"))

display(inflation_df)

year,inflation_rate
1960,1.779877847
1961,1.695212939
1962,3.632214971
1963,2.946161357
1964,13.35526115
1965,9.474758592
1966,10.80184835
1967,13.06220248
1968,3.237412426
1969,-0.58413661


In [0]:
interest_df = interest_df.withColumn(
    "date",
    to_date(col("observation_date"), "dd-MM-yyyy")
)

interest_df = interest_df.withColumn(
    "year",
    year(col("date"))
)

interest_df = interest_df.withColumnRenamed("Interest rate", "interest_rate")

In [0]:
interest_df = interest_df.groupBy("year") \
    .agg(avg("interest_rate").alias("interest_rate"))

display(interest_df)

year,interest_rate
2011,8.56
2012,8.259433888916666
2013,8.109994437500001
2014,8.586742083333332
2015,7.77597097225
2016,7.206195000000001
2017,6.923906666666666
2018,7.704654166666668
2019,6.9969987499999995
2020,6.191779166666667


In [0]:
econ_df = pop_df.join(inflation_df, "year") \
                .join(interest_df, "year") \
                .orderBy("year")

display(econ_df)

year,population_growth,inflation_rate,interest_rate
2011,1.37%,8.911793365,8.56
2012,1.34%,9.478996914,8.259433888916666
2013,1.31%,10.01787847,8.109994437500001
2014,1.25%,6.665656719,8.586742083333332
2015,1.19%,4.906973441,7.77597097225
2016,1.19%,4.948216341,7.206195000000001
2017,1.16%,3.328173375,6.923906666666666
2018,1.09%,3.938826467,7.704654166666668
2019,1.03%,3.729505735,6.9969987499999995
2020,0.96%,6.623436776,6.191779166666667


In [0]:
%pip install india-housing-datasets

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
from india_housing_datasets import load_housing
df_pandas = load_housing('delhi')
df = spark.createDataFrame(df_pandas)
display(df)

city,locality,area_sqft,bhk,bath,floor,age_years,price_lakhs
delhi,Vasant Kunj,1200,2,2,2,10,145.5
delhi,Rohini Sector 13,850,2,2,3,15,65.2
delhi,Dwarka Sector 6,1450,3,3,5,8,125.0
delhi,South Extension,1800,3,3,1,12,310.0
delhi,Janakpuri,1100,2,2,0,20,95.5
delhi,Greater Kailash,2400,4,4,2,5,550.0
delhi,Laxmi Nagar,650,2,1,2,12,38.0
delhi,Pitampura,1350,3,2,4,10,155.0
delhi,Uttam Nagar,450,1,1,1,5,22.4
delhi,Saket,1600,3,3,3,7,210.0


In [0]:
from pyspark.sql.functions import monotonically_increasing_id
df = df.withColumn(
    "year",
    (monotonically_increasing_id() % 12) + 2011
)

In [0]:
df = df.select(
    "area_sqft",
    "bhk",
    "bath",
    "price_lakhs",
    "locality",
    "year"
)

In [0]:
final_df = df.join(econ_df, "year", "inner")
display(final_df)

year,area_sqft,bhk,bath,price_lakhs,locality,population_growth,inflation_rate,interest_rate
2011,750,2,1,32.0,Baprola,1.37%,8.911793365,8.56
2012,800,2,2,35.0,Hiran Kudna,1.34%,9.478996914,8.259433888916666
2013,700,2,1,30.0,Tikri Kalan,1.31%,10.01787847,8.109994437500001
2014,950,3,2,42.0,Bakkarwala,1.25%,6.665656719,8.586742083333332
2015,1050,3,2,55.0,Hastsal,1.19%,4.906973441,7.77597097225
2016,1150,3,3,68.0,Bindapur,1.19%,4.948216341,7.206195000000001
2017,1250,3,2,75.0,Matiala,1.16%,3.328173375,6.923906666666666
2018,950,2,2,82.0,Paschim Puri,1.09%,3.938826467,7.704654166666668
2019,700,2,1,45.0,Madipur,1.03%,3.729505735,6.9969987499999995
2020,1000,2,2,65.0,Peera Garhi,0.96%,6.623436776,6.191779166666667


In [0]:
years_df = econ_df.select("year").distinct()

In [0]:
df = df.drop("year")

In [0]:
housing_expanded = df.crossJoin(years_df)

In [0]:
final_df = housing_expanded.join(econ_df, "year", "inner") \
                          .orderBy("year", "locality")

display(final_df)

year,area_sqft,bhk,bath,price_lakhs,locality,population_growth,inflation_rate,interest_rate
2011,1000,3,2,58.0,Abul Fazal,1.37%,8.911793365,8.56
2011,1200,3,3,75.0,Abul Fazal Enclave,1.37%,8.911793365,8.56
2011,1200,3,2,115.0,Adarsh Nagar,1.37%,8.911793365,8.56
2011,1000,3,2,92.0,Adchini,1.37%,8.911793365,8.56
2011,4000,6,6,5500.0,Akbar Road,1.37%,8.911793365,8.56
2011,1450,3,3,165.0,Alaknanda,1.37%,8.911793365,8.56
2011,1200,2,2,118.0,Alaknanda,1.37%,8.911793365,8.56
2011,950,2,2,115.0,Amar Colony,1.37%,8.911793365,8.56
2011,850,2,1,55.0,Amberhai,1.37%,8.911793365,8.56
2011,1400,3,2,155.0,Ambica Vihar,1.37%,8.911793365,8.56


In [0]:
from pyspark.sql.functions import col

final_df = final_df.withColumn(
    "price_per_sqft",
    col("price_lakhs") * 100000 / col("area_sqft")
)

In [0]:
from pyspark.sql.functions import regexp_replace

final_df = final_df.withColumn(
    "population_growth",
    regexp_replace("population_growth", "%", "")
)

In [0]:
from pyspark.sql.functions import col

final_df = final_df.withColumn(
    "population_growth",
    col("population_growth").cast("double")
)

In [0]:
final_df.printSchema()

root
 |-- year: long (nullable = true)
 |-- area_sqft: long (nullable = true)
 |-- bhk: long (nullable = true)
 |-- bath: long (nullable = true)
 |-- price_lakhs: double (nullable = true)
 |-- locality: string (nullable = true)
 |-- population_growth: double (nullable = true)
 |-- inflation_rate: double (nullable = true)
 |-- interest_rate: double (nullable = true)
 |-- price_per_sqft: double (nullable = true)



In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

econ_features = ["population_growth", "inflation_rate", "interest_rate"]

assembler = VectorAssembler(inputCols=econ_features, outputCol="econ_vector")

temp_df = assembler.transform(final_df)

scaler = StandardScaler(inputCol="econ_vector", outputCol="scaled_econ", withStd=True, withMean=True)

scaler_model = scaler.fit(temp_df)
temp_df = scaler_model.transform(temp_df)

In [0]:
from pyspark.sql.functions import expr

temp_df = temp_df.withColumn(
    "risk_score",
    expr("0.5 * inflation_rate + 0.3 * interest_rate - 0.2 * population_growth")
)

In [0]:
from pyspark.sql.functions import when

temp_df = temp_df.withColumn(
    "investment_label",
    when(col("risk_score") < 3, "LOW_RISK")
    .when(col("risk_score") < 6, "MEDIUM_RISK")
    .otherwise("HIGH_RISK")
)

In [0]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol="locality", outputCol="locality_index")

temp_df = indexer.fit(temp_df).transform(temp_df)

In [0]:
feature_cols = [
    "area_sqft",
    "bhk",
    "bath",
    "price_per_sqft",
    "locality_index",
    "population_growth",
    "inflation_rate",
    "interest_rate"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

final_df = assembler.transform(temp_df)

**PRICE PREDICTION**


In [0]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [
    "area_sqft",
    "bhk",
    "bath",
    "price_per_sqft",
    "population_growth",
    "inflation_rate",
    "interest_rate"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

final_df = assembler.transform(temp_df)

In [0]:
from pyspark.ml.regression import (
    LinearRegression,
    DecisionTreeRegressor,
    RandomForestRegressor,
    GBTRegressor,
    GeneralizedLinearRegression
)
from pyspark.ml.evaluation import RegressionEvaluator

# Train-test split
train_df, test_df = final_df.randomSplit([0.8, 0.2], seed=42)

# Evaluator
evaluator = RegressionEvaluator(
    labelCol="price_lakhs",
    predictionCol="prediction",
    metricName="rmse"
)

results = {}

# 1. Linear Regression
lr = LinearRegression(featuresCol="features", labelCol="price_lakhs")
lr_model = lr.fit(train_df)
lr_pred = lr_model.transform(test_df)
results["Linear Regression"] = evaluator.evaluate(lr_pred)

# 2. Decision Tree
dt = DecisionTreeRegressor(featuresCol="features", labelCol="price_lakhs")
dt_model = dt.fit(train_df)
dt_pred = dt_model.transform(test_df)
results["Decision Tree"] = evaluator.evaluate(dt_pred)

# 3. Random Forest
rf = RandomForestRegressor(featuresCol="features", labelCol="price_lakhs", numTrees=50)
rf_model = rf.fit(train_df)
rf_pred = rf_model.transform(test_df)
results["Random Forest"] = evaluator.evaluate(rf_pred)

# 4. GBT Regressor
gbt = GBTRegressor(featuresCol="features", labelCol="price_lakhs", maxIter=50)
gbt_model = gbt.fit(train_df)
gbt_pred = gbt_model.transform(test_df)
results["GBT Regressor"] = evaluator.evaluate(gbt_pred)

# 5. Generalized Linear Regression
glr = GeneralizedLinearRegression(featuresCol="features", labelCol="price_lakhs")
glr_model = glr.fit(train_df)
glr_pred = glr_model.transform(test_df)
results["Generalized Linear Regression"] = evaluator.evaluate(glr_pred)

# Results
print("Model Performance (RMSE):")
for model, score in results.items():
    print(f"{model}: {score}")

Model Performance (RMSE):
Linear Regression: 111.51752405585776
Decision Tree: 291.75367100201106
Random Forest: 253.85503317783582
GBT Regressor: 323.7281436517933
Generalized Linear Regression: 111.51752405585776


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse_evaluator = RegressionEvaluator(
    labelCol="price_lakhs",
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol="price_lakhs",
    predictionCol="prediction",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol="price_lakhs",
    predictionCol="prediction",
    metricName="r2"
)

models_predictions = {
    "Linear Regression": lr_pred,
    "Decision Tree": dt_pred,
    "Random Forest": rf_pred,
    "GBT Regressor": gbt_pred,
    "GLR": glr_pred
}

print("Model Comparison:\n")

for model_name, pred_df in models_predictions.items():
    rmse = rmse_evaluator.evaluate(pred_df)
    mae = mae_evaluator.evaluate(pred_df)
    r2 = r2_evaluator.evaluate(pred_df)
    
    print(f"{model_name}")
    print(f" RMSE: {rmse}")
    print(f" MAE : {mae}")
    print(f" R2  : {r2}")
    print("-" * 30)

Model Comparison:

Linear Regression
 RMSE: 111.51752405585776
 MAE : 80.93279545373885
 R2  : 0.9819447381240585
------------------------------
Decision Tree
 RMSE: 291.75367100201106
 MAE : 73.83989929517615
 R2  : 0.8764194244910544
------------------------------
Random Forest
 RMSE: 253.85503317783582
 MAE : 61.07028226776697
 R2  : 0.9064402372263112
------------------------------
GBT Regressor
 RMSE: 323.7281436517933
 MAE : 57.235428999442675
 R2  : 0.8478477186090232
------------------------------
GLR
 RMSE: 111.51752405585776
 MAE : 80.93279545373885
 R2  : 0.9819447381240585
------------------------------


In [0]:
from pyspark.sql.functions import avg

avg_vals = temp_df.select(
    avg("population_growth").alias("population_growth"),
    avg("inflation_rate").alias("inflation_rate"),
    avg("interest_rate").alias("interest_rate"),
    avg("area_sqft").alias("area_sqft"),
    avg("bhk").alias("bhk"),
    avg("bath").alias("bath"),
    avg("price_per_sqft").alias("price_per_sqft")
).collect()[0]

In [0]:
future_years = list(range(2025, 2036))

future_data = []

for year in future_years:
    future_data.append((
        year,
        float(avg_vals["area_sqft"]),
        float(avg_vals["bhk"]),
        float(avg_vals["bath"]),
        float(avg_vals["price_per_sqft"]),
        float(avg_vals["population_growth"]),
        float(avg_vals["inflation_rate"]),
        float(avg_vals["interest_rate"])
    ))

In [0]:
columns = [
    "year",
    "area_sqft",
    "bhk",
    "bath",
    "price_per_sqft",
    "population_growth",
    "inflation_rate",
    "interest_rate"
]

future_df = spark.createDataFrame(future_data, columns)

In [0]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [
    "area_sqft",
    "bhk",
    "bath",
    "price_per_sqft",
    "population_growth",
    "inflation_rate",
    "interest_rate"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

future_df = assembler.transform(future_df)

In [0]:
future_predictions = lr_model.transform(future_df)

display(future_predictions.select("year", "prediction"))

year,prediction
2025,390.87643589623633
2026,390.87643589623633
2027,390.87643589623633
2028,390.87643589623633
2029,390.87643589623633
2030,390.87643589623633
2031,390.87643589623633
2032,390.87643589623633
2033,390.87643589623633
2034,390.87643589623633


In [0]:
future_data = []

for i, year in enumerate(range(2025, 2036)):
    
    growth_factor = 1 + (i * 0.03)   # 3% yearly growth
    
    future_data.append((
        year,
        float(avg_vals["area_sqft"]),
        float(avg_vals["bhk"]),
        float(avg_vals["bath"]),
        float(avg_vals["price_per_sqft"]) * growth_factor,
        float(avg_vals["population_growth"]) * (1 + i*0.01),
        float(avg_vals["inflation_rate"]) * (1 + i*0.02),
        float(avg_vals["interest_rate"]) * (1 + i*0.015)
    ))

In [0]:
localities = df.select("locality").distinct()

In [0]:
from pyspark.sql import Row

years_df = spark.createDataFrame([(y,) for y in range(2025, 2036)], ["year"])

future_base = localities.crossJoin(years_df)

In [0]:
from pyspark.sql.functions import lit

future_df = future_base \
    .withColumn("area_sqft", lit(avg_vals["area_sqft"])) \
    .withColumn("bhk", lit(avg_vals["bhk"])) \
    .withColumn("bath", lit(avg_vals["bath"])) \
    .withColumn("price_per_sqft", lit(avg_vals["price_per_sqft"])) \
    .withColumn("population_growth", lit(avg_vals["population_growth"])) \
    .withColumn("inflation_rate", lit(avg_vals["inflation_rate"])) \
    .withColumn("interest_rate", lit(avg_vals["interest_rate"]))

In [0]:
from pyspark.sql.functions import col

future_df = future_df.withColumn(
    "price_per_sqft",
    col("price_per_sqft") * (1 + (col("year") - 2025) * 0.03)
)

In [0]:
assembler = VectorAssembler(
    inputCols=[
        "area_sqft",
        "bhk",
        "bath",
        "price_per_sqft",
        "population_growth",
        "inflation_rate",
        "interest_rate"
    ],
    outputCol="features"
)

future_df = assembler.transform(future_df)

In [0]:
localities = df.select("locality").distinct()

In [0]:
years_df = spark.createDataFrame([(y,) for y in range(2025, 2036)], ["year"])

In [0]:
future_base = years_df.crossJoin(localities)

In [0]:
from pyspark.sql.functions import lit

future_df = future_base \
    .withColumn("area_sqft", lit(avg_vals["area_sqft"])) \
    .withColumn("bhk", lit(avg_vals["bhk"])) \
    .withColumn("bath", lit(avg_vals["bath"])) \
    .withColumn("price_per_sqft", lit(avg_vals["price_per_sqft"])) \
    .withColumn("population_growth", lit(avg_vals["population_growth"])) \
    .withColumn("inflation_rate", lit(avg_vals["inflation_rate"])) \
    .withColumn("interest_rate", lit(avg_vals["interest_rate"]))

In [0]:
from pyspark.sql.functions import col

future_df = future_df.withColumn(
    "price_per_sqft",
    col("price_per_sqft") * (1 + (col("year") - 2025) * 0.03)
)

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "area_sqft",
        "bhk",
        "bath",
        "price_per_sqft",
        "population_growth",
        "inflation_rate",
        "interest_rate"
    ],
    outputCol="features"
)

future_df = assembler.transform(future_df)

In [0]:
future_predictions = lr_model.transform(future_df)

In [0]:
from pyspark.sql.functions import col

display(
    future_predictions
    .select("year", "locality", "prediction")
    .orderBy(col("year"), col("locality"))
)

year,locality,prediction
2025,Abul Fazal,390.87643589623633
2025,Abul Fazal Enclave,390.87643589623633
2025,Adarsh Nagar,390.87643589623633
2025,Adchini,390.87643589623633
2025,Akbar Road,390.87643589623633
2025,Alaknanda,390.87643589623633
2025,Amar Colony,390.87643589623633
2025,Amberhai,390.87643589623633
2025,Ambica Vihar,390.87643589623633
2025,Amrit Vihar,390.87643589623633


In [0]:
from pyspark.ml.feature import OneHotEncoder

encoder = OneHotEncoder(
    inputCols=["locality_index"],
    outputCols=["locality_vec"]
)

temp_df = encoder.fit(temp_df).transform(temp_df)

In [0]:
feature_cols = [
    "area_sqft",
    "bhk",
    "bath",
    "price_per_sqft",
    "population_growth",
    "inflation_rate",
    "interest_rate",
    "locality_vec"   
]

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

final_df = assembler.transform(temp_df)

In [0]:
from pyspark.ml.regression import LinearRegression

train_df, test_df = final_df.randomSplit([0.8, 0.2], seed=42)

lr = LinearRegression(featuresCol="features", labelCol="price_lakhs")
lr_model = lr.fit(train_df)

In [0]:
from pyspark.sql.functions import avg

locality_prices = df.groupBy("locality").agg(
    avg("price_per_sqft").alias("price_per_sqft")
)

future_df = future_df.join(locality_prices, "locality", "left")

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "area_sqft",
        "bhk",
        "bath",
        "price_per_sqft",
        "population_growth",
        "inflation_rate",
        "interest_rate",
        "locality_vec"
    ],
    outputCol="features"
)

future_df = assembler.transform(future_df)

In [0]:
from pyspark.sql.functions import col, avg

# Add price_per_sqft to df
df = df.withColumn(
    "price_per_sqft",
    col("price_lakhs") * 100000 / col("area_sqft")
)

# Aggregate by locality
locality_prices = df.groupBy("locality").agg(
    avg("price_per_sqft").alias("price_per_sqft")
)

# Join to future_df
future_df = future_df.join(locality_prices, "locality", "left")

future_df.printSchema()

root
 |-- locality: string (nullable = true)
 |-- year: long (nullable = true)
 |-- area_sqft: double (nullable = false)
 |-- bhk: double (nullable = false)
 |-- bath: double (nullable = false)
 |-- price_per_sqft: double (nullable = true)
 |-- population_growth: double (nullable = false)
 |-- inflation_rate: double (nullable = false)
 |-- interest_rate: double (nullable = false)
 |-- features: vectorudt (nullable = true)
 |-- price_per_sqft: double (nullable = true)



In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "area_sqft",
        "bhk",
        "bath",
        "price_per_sqft",
        "population_growth",
        "inflation_rate",
        "interest_rate",
        "locality_vec"
    ],
    outputCol="features"
)

future_df = assembler.transform(future_df)

In [0]:
from pyspark.sql.functions import when, col

future_predictions = future_predictions.withColumn(
    "prediction",
    when(col("prediction") < 0, 0).otherwise(col("prediction"))
)

In [0]:
future_predictions = future_predictions.withColumn(
    "prediction",
    when(col("prediction") < 50, 50).otherwise(col("prediction"))
)

In [0]:
from pyspark.sql.functions import col, avg, lit, log, exp, when
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression

# 1. FEATURE ENGINEERING 
df_clean = df.select("area_sqft", "bhk", "bath", "price_lakhs", "locality")

df_clean = df_clean.withColumn(
    "price_per_sqft",
    col("price_lakhs") * 100000 / col("area_sqft")
)

# LOG TARGET
df_clean = df_clean.withColumn("log_price", log(col("price_lakhs")))

# 2. ENCODING

indexer = StringIndexer(inputCol="locality", outputCol="locality_index")
indexer_model = indexer.fit(df_clean)
df_clean = indexer_model.transform(df_clean)

encoder = OneHotEncoder(inputCols=["locality_index"], outputCols=["locality_vec"])
encoder_model = encoder.fit(df_clean)
df_clean = encoder_model.transform(df_clean)


# 3. FEATURES

feature_cols = [
    "area_sqft",
    "bhk",
    "bath",
    "price_per_sqft",
    "locality_vec"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

final_df = assembler.transform(df_clean)


# 4. TRAIN MODEL

train_df, test_df = final_df.randomSplit([0.8, 0.2], seed=42)

lr = LinearRegression(featuresCol="features", labelCol="log_price")
lr_model = lr.fit(train_df)


# 5. FUTURE DATA

years_df = spark.createDataFrame([(y,) for y in range(2025, 2036)], ["year"])
localities_df = df_clean.select("locality").distinct()

future_df = years_df.crossJoin(localities_df)

avg_vals = df_clean.select(
    avg("area_sqft").alias("area_sqft"),
    avg("bhk").alias("bhk"),
    avg("bath").alias("bath"),
    avg("price_per_sqft").alias("price_per_sqft")
).collect()[0]

future_df = future_df \
    .withColumn("area_sqft", lit(avg_vals["area_sqft"])) \
    .withColumn("bhk", lit(avg_vals["bhk"])) \
    .withColumn("bath", lit(avg_vals["bath"])) \
    .withColumn("price_per_sqft", lit(avg_vals["price_per_sqft"]))

future_df = future_df.withColumn(
    "price_per_sqft",
    col("price_per_sqft") * (1 + (col("year") - 2025) * 0.03)
)


# 6. APPLY ENCODERS

future_df = indexer_model.transform(future_df)
future_df = encoder_model.transform(future_df)


# 7. FEATURES

future_df = assembler.transform(future_df)


# 8. PREDICT

future_predictions = lr_model.transform(future_df)

# convert log → real price
future_predictions = future_predictions.withColumn(
    "prediction",
    exp(col("prediction"))
)


# 9. FILTER OUT INVALID VALUES 

future_predictions = future_predictions.filter(
    (col("prediction").isNotNull()) &
    (col("prediction") > 0)
)

# 10. FINAL OUTPUT

display(
    future_predictions
    .select("year", "locality", "prediction")
    .orderBy("year", "locality")
)

year,locality,prediction
2025,Abul Fazal,96.43733097983554
2025,Abul Fazal Enclave,102.40084633476687
2025,Adarsh Nagar,161.94934706263717
2025,Adchini,148.62076866696634
2025,Akbar Road,288.37373150466675
2025,Alaknanda,202.96360932904437
2025,Amar Colony,202.96360932904437
2025,Amberhai,110.0186847469191
2025,Ambica Vihar,188.50926318307236
2025,Amrit Vihar,85.11130542994924


CLASSIFICATION

In [0]:
classified_df = future_predictions.select(
    "year",
    "locality",
    "prediction"
)

In [0]:
from pyspark.sql.functions import when

classified_df = classified_df.withColumn(
    "price_category",
    when(col("prediction") < 100, "LOW_VALUE")
    .when(col("prediction") < 200, "MEDIUM_VALUE")
    .otherwise("HIGH_VALUE")
)

In [0]:
quantiles = classified_df.approxQuantile("prediction", [0.33, 0.66], 0)

low_threshold = quantiles[0]
high_threshold = quantiles[1]

In [0]:
classified_df = classified_df.withColumn(
    "price_category",
    when(col("prediction") < low_threshold, "LOW_VALUE")
    .when(col("prediction") < high_threshold, "MEDIUM_VALUE")
    .otherwise("HIGH_VALUE")
)

In [0]:
display(
    classified_df
    .filter("price_category = 'LOW_VALUE'")
    .select("year", "locality", "prediction")
    .orderBy("year", "locality")
)

year,locality,prediction
2025,Abul Fazal,96.43733097983554
2025,Abul Fazal Enclave,102.40084633476687
2025,Amberhai,110.0186847469191
2025,Amrit Vihar,85.11130542994924
2025,Anand Gram,67.24417322315117
2025,Arvind Nagar,82.25423373271491
2025,Aya Nagar,69.7383565123593
2025,Babarpur,83.07402153020837
2025,Badarpur,72.86787258294565
2025,Bagdola,102.73159488923602


In [0]:
display(
    classified_df
    .filter("price_category = 'HIGH_VALUE'")
    .select("year", "locality", "prediction")
    .orderBy("year", "locality")
)

year,locality,prediction
2025,Akbar Road,288.37373150466675
2025,Amrita Shergill Marg,259.8257395599622
2025,Anand Niketan,285.3883060942207
2025,Ashok Vihar Phase 2,215.065910450483
2025,Ashok Vihar Phase 3,220.96224155533451
2025,August Kranti Marg,341.64790848515526
2025,Bali Nagar,216.54452317176026
2025,Bapa Nagar,255.5698003880676
2025,Barakhamba Road,478.71167335828585
2025,Bela Road,247.18177683726347


In [0]:
display(
    classified_df
    .groupBy("year", "price_category")
    .count()
    .orderBy("year", "price_category")
)

year,price_category,count
2025,HIGH_VALUE,113
2025,LOW_VALUE,150
2025,MEDIUM_VALUE,178
2026,HIGH_VALUE,114
2026,LOW_VALUE,150
2026,MEDIUM_VALUE,177
2027,HIGH_VALUE,116
2027,LOW_VALUE,149
2027,MEDIUM_VALUE,176
2028,HIGH_VALUE,118


In [0]:
from pyspark.sql.functions import col

sorted_df = classified_df.orderBy("year", "locality")

In [0]:
sorted_df.write.mode("overwrite") \
    .saveAsTable("workspace.default.delhi_real_estate_final")